# Retrieval-Augmented Generation (RAG) — From First Principles

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/simple_rag.ipynb)

**A complete, first-principles guide to building RAG systems using native Python.**

This notebook is the foundational reference for Retrieval-Augmented Generation. Every concept is built from scratch — no LangChain, no LlamaIndex, no paid APIs. By the end, you will understand *exactly* how each component works and why.

| Component | Implementation |
|---|---|
| **LLM** | Qwen/Qwen3-8B (4-bit NF4 quantization) |
| **Embeddings** | BAAI/bge-base-en-v1.5 (768-dim, sentence-transformers) |
| **Vector Store** | FAISS (faiss-cpu) with numpy arrays |
| **Data** | Synthetic inline knowledge base (no external files) |
| **Framework** | Pure Python — zero abstraction layers |

> **Runtime:** Google Colab with T4 GPU. Estimated setup time: ~3 minutes.

## 1.1 — What is RAG and Why Does It Exist?

Large Language Models (LLMs) are trained on a fixed snapshot of text data. Once training ends, the model's knowledge is frozen — it cannot learn new facts, correct outdated information, or access proprietary data it never saw. This creates three fundamental problems:

1. **Knowledge Cutoff**: An LLM trained through January 2024 cannot answer questions about events in March 2024. The model doesn't "know" the information simply doesn't exist in its parameter weights.

2. **Hallucination**: When asked about facts outside its training data — or even sometimes within it — LLMs confidently generate plausible-sounding but factually incorrect text. The model has no mechanism to distinguish "I know this" from "I'm pattern-matching to something that sounds right."

3. **No Attribution**: Even when an LLM provides a correct answer, it cannot point to a source. In domains like medicine, law, and finance, unsourced claims are essentially useless.

**Retrieval-Augmented Generation (RAG)** solves all three problems with a simple but powerful idea: instead of relying solely on the model's parametric memory, we *retrieve* relevant documents from an external knowledge base and inject them directly into the prompt. The LLM then generates its answer grounded in the retrieved evidence.

$$\text{RAG}(q) = \text{Generate}\big(q, \; \text{Retrieve}(q, \mathcal{D})\big)$$

where $q$ is the user query and $\mathcal{D}$ is the document collection.

## 1.2 — The RAG Architecture

At its core, every RAG system follows the same four-stage pipeline:

```
┌─────────────────────────────────────────────────────────────────┐
│                     RAG Pipeline                                │
│                                                                 │
│   ┌─────────┐    ┌───────────┐    ┌─────────┐    ┌──────────┐  │
│   │  QUERY  │───▶│ RETRIEVE  │───▶│ AUGMENT │───▶│ GENERATE │  │
│   │         │    │           │    │         │    │          │  │
│   │ "What   │    │ Search    │    │ Build   │    │ LLM uses │  │
│   │  causes │    │ vector DB │    │ prompt  │    │ context  │  │
│   │  ocean  │    │ for top-k │    │ with    │    │ to write │  │
│   │  acid.?"│    │ matches   │    │ context │    │ answer   │  │
│   └─────────┘    └───────────┘    └─────────┘    └──────────┘  │
│                                                                 │
│   OFFLINE (Indexing):                                           │
│   Documents ──▶ Chunk ──▶ Embed ──▶ Store in Vector DB          │
└─────────────────────────────────────────────────────────────────┘
```

**Offline stage** (done once): We take our documents, split them into chunks, compute embeddings for each chunk, and store them in a vector index.

**Online stage** (per query): The user's question is embedded, compared against all stored chunk embeddings, the top-k most similar chunks are retrieved, injected into a prompt, and sent to the LLM for generation.

This separation of concerns — retrieval vs. generation — is what makes RAG so powerful and modular.

## 1.3 — Historical Context: From Open-Book QA to RAG

RAG didn't appear in a vacuum. It sits at the intersection of two research traditions:

**Information Retrieval (IR):** For decades, search engines used keyword-based methods — TF-IDF, BM25 — to find relevant documents. These methods are fast and surprisingly effective but fundamentally *lexical*: they match words, not meaning. The query "automobile fuel efficiency" won't match a document about "car miles per gallon" unless the exact words overlap.

**Neural Question Answering:** Starting around 2017, researchers began using neural networks to answer questions given a context passage (extractive QA). Models like BERT could read a paragraph and highlight the answer span. But they needed someone to *find* the right paragraph first.

**The synthesis — RAG — was formalized by Lewis et al. (2020)** in their paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." Their key insight was to make retrieval *differentiable* — training the retriever and generator end-to-end. While modern production RAG systems typically use frozen retrievers (as we do here), the paper established the paradigm:

> *"Rather than storing all knowledge in the model's parameters, augment the model with a non-parametric memory (a retrieval corpus) that can be updated without retraining."*

This idea has become the dominant architecture for knowledge-intensive AI applications in 2024–2025.

## 1.4 — When RAG Beats Fine-Tuning

A common alternative to RAG is **fine-tuning**: updating the model's weights on domain-specific data. Both approaches inject external knowledge, but they differ in fundamental ways:

| Dimension | RAG | Fine-Tuning |
|---|---|---|
| **Knowledge freshness** | Instant — update the document store | Requires retraining (hours/days) |
| **Cost** | Cheap — no GPU training needed | Expensive — GPU hours + data prep |
| **Attribution** | Built-in — retrieved chunks are citeable | None — knowledge is baked into weights |
| **Hallucination control** | Grounded in retrieved text | Can still hallucinate freely |
| **Domain adaptation** | Moderate — retriever must find relevant docs | Strong — model internalizes domain patterns |
| **Latency** | Higher — retrieval adds ~50-200ms | Lower — single forward pass |

**Use RAG when:** You need factual accuracy, source attribution, frequently changing data, or you can't afford to retrain. This covers *most* enterprise use cases.

**Use fine-tuning when:** You need the model to adopt a specific style, learn domain-specific reasoning patterns, or handle tasks where retrieval isn't applicable (e.g., code generation in a specific framework).

**Use both when:** You want the model to reason in your domain's style AND have access to up-to-date facts. This combination — sometimes called RAG + LoRA — is increasingly common in production.

## 1.5 — The Information Retrieval Pipeline

To understand RAG deeply, we need to understand how retrieval has evolved:

**Classical IR (TF-IDF, BM25):** Documents and queries are represented as sparse vectors of word frequencies. Similarity is computed via dot product. Strengths: fast, interpretable, no training needed. Weakness: purely lexical — misses synonyms and paraphrases.

**Neural IR (Dense Retrieval):** Documents and queries are encoded into dense vectors (embeddings) by neural networks. Similarity is computed via cosine similarity or dot product in embedding space. Strengths: captures semantic meaning ("car" ≈ "automobile"). Weakness: requires a trained encoder, computationally heavier.

**RAG combines neural IR with generative AI:** The retriever finds relevant passages using dense embeddings, and the generator synthesizes a natural language answer from those passages. This is fundamentally different from traditional search engines, which merely *rank* documents — RAG *reads* them and *answers* questions.

In this notebook, we implement the full neural IR + generation pipeline from scratch.

---
# Part 2 — Environment Setup

We install all dependencies, load our models, and define helper functions. Everything runs on a free Google Colab T4 GPU.

In [ ]:
# ============================================================
# Cell 1: Install all dependencies
# ============================================================
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

In [ ]:
# ============================================================
# Cell 2: Mount Google Drive & Load Qwen3-8B (4-bit quantized)
# ============================================================
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print(f"Model loaded: {MODEL_NAME}")
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ============================================================
# Cell 3: Load BGE Embedding Model
# ============================================================
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print(f"Embedding model loaded: BAAI/bge-base-en-v1.5")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {embed_model.max_seq_length}")

In [ ]:
# ============================================================
# Cell 4: Define the generation helper
# ============================================================
def generate(prompt, max_new_tokens=512):
    """Generate text using Qwen3-8B with chat template."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        top_k=20, temperature=0.7
    )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

# Quick test
response = generate("What is 2 + 2? Answer in one sentence.")
print(f"LLM test: {response}")

---
# Part 3 — Understanding Embeddings

Embeddings are the foundation of modern retrieval. Before we can build a RAG system, we need to deeply understand what embeddings are, how they work, and why they enable semantic search.

## 3.1 — What Are Embeddings?

An **embedding** is a mapping from discrete, variable-length text into a fixed-size continuous vector in $\mathbb{R}^d$. The key property is that *semantically similar texts are mapped to nearby points* in the vector space.

Formally, an embedding function $E$ maps a text string $s$ to a dense vector:

$$E: \text{String} \rightarrow \mathbb{R}^d$$

For the BGE model we're using, $d = 768$, so every piece of text — whether it's a single word or an entire paragraph — gets mapped to a point in 768-dimensional space.

**Why is this useful?** Because once text is represented as vectors, we can use standard mathematical operations to measure similarity, cluster documents, and search efficiently. The entire field of vector databases and semantic search is built on this idea.

The magic is in the training: embedding models like BGE are trained on millions of text pairs (e.g., questions and their answers, paraphrases, entailment pairs) so that the model learns to place related texts close together and unrelated texts far apart.

In [ ]:
# ============================================================
# Cell: Examine a raw embedding
# ============================================================
import numpy as np

text = "Climate change affects global ecosystems"
embedding = embed_model.encode(text)

print(f"Text: '{text}'")
print(f"Embedding type: {type(embedding)}")
print(f"Embedding shape: {embedding.shape}")      # (768,)
print(f"Embedding dtype: {embedding.dtype}")
print(f"\nFirst 10 dimensions:")
for i, val in enumerate(embedding[:10]):
    print(f"  dim[{i}] = {val:+.6f}")
print(f"\nVector statistics:")
print(f"  Min:  {embedding.min():.6f}")
print(f"  Max:  {embedding.max():.6f}")
print(f"  Mean: {embedding.mean():.6f}")
print(f"  Std:  {embedding.std():.6f}")
print(f"  Norm: {np.linalg.norm(embedding):.6f}")

## 3.2 — Why 768 Dimensions?

Why not 10 dimensions? Or 10,000? The choice of embedding dimensionality involves a fundamental tradeoff:

**Too few dimensions (e.g., 32):** The model cannot capture enough nuance. Unrelated texts get mapped to similar vectors because there aren't enough "axes" to distinguish them. This is related to the **Johnson-Lindenstrauss lemma** — you need $O(\log n / \epsilon^2)$ dimensions to preserve pairwise distances among $n$ points.

**Too many dimensions (e.g., 4096):** Storage costs scale linearly with $d$. Search time also increases. And there's a subtler problem: the **curse of dimensionality** — in very high-dimensional spaces, all points tend to be roughly equidistant, making nearest-neighbor search less meaningful.

**768 dimensions** (used by BERT-base and BGE-base) is an empirically validated sweet spot. It's large enough to capture rich semantic relationships across diverse domains, yet small enough for efficient storage and search. For reference:
- 1 million documents × 768 dimensions × 4 bytes = **~2.9 GB** of index storage
- This fits comfortably in a Colab T4's memory

Larger models like BGE-large use 1024 dimensions for marginally better quality at higher cost.

In [ ]:
# ============================================================
# Cell: Cosine similarity from scratch
# ============================================================
import numpy as np

def cosine_similarity(a, b):
    """
    Compute cosine similarity between two vectors.

    cos(a, b) = (a · b) / (||a|| × ||b||)

    Returns a value in [-1, 1]:
      1.0  = identical direction (most similar)
      0.0  = orthogonal (unrelated)
     -1.0  = opposite direction (most dissimilar)
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

# Demonstrate with simple 2D vectors
v1 = np.array([1.0, 0.0])
v2 = np.array([0.0, 1.0])
v3 = np.array([1.0, 1.0])
v4 = np.array([-1.0, 0.0])

print("2D vector examples:")
print(f"  cos([1,0], [0,1])  = {cosine_similarity(v1, v2):.4f}  (orthogonal)")
print(f"  cos([1,0], [1,1])  = {cosine_similarity(v1, v3):.4f}  (45° angle)")
print(f"  cos([1,0], [-1,0]) = {cosine_similarity(v1, v4):.4f}  (opposite)")
print(f"  cos([1,0], [1,0])  = {cosine_similarity(v1, v1):.4f}  (identical)")

In [ ]:
# ============================================================
# Cell: Semantic similarity with real embeddings
# ============================================================
sentences = [
    "The cat sat on the mat",                # 0: animals
    "A kitten rested on the rug",            # 1: animals (paraphrase of 0)
    "Stock prices rose sharply today",       # 2: finance
    "The equity market surged this morning", # 3: finance (paraphrase of 2)
    "Quantum computing uses qubits",         # 4: tech (unrelated)
]

embeddings = embed_model.encode(sentences)

print("Pairwise cosine similarity matrix:")
print(f"{'':>45}", end="")
for j in range(len(sentences)):
    print(f"  [{j}]  ", end="")
print()

for i in range(len(sentences)):
    label = sentences[i][:42].ljust(45)
    print(f"{label}", end="")
    for j in range(len(sentences)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{sim:.3f}  ", end="")
    print()

print("\nKey observations:")
print("  • Sentences 0 & 1 (cat/kitten) — high similarity despite different words")
print("  • Sentences 2 & 3 (stocks/equity) — high similarity (paraphrases)")
print("  • Sentences 0 & 4 (cat vs quantum) — low similarity (unrelated topics)")

## 3.3 — Semantic vs. Lexical Similarity

The results above reveal something profound: **embeddings capture meaning, not surface-level word overlap.**

Consider sentences 0 and 1:
- *"The cat sat on the mat"*
- *"A kitten rested on the rug"*

A keyword-based system (like BM25) would find low overlap — only "on" and "the" match. But the embedding model recognizes that "cat" ≈ "kitten", "sat" ≈ "rested", and "mat" ≈ "rug" at a semantic level. This is what enables semantic search.

This property emerges from the model's training on massive text corpora. During training, the model learns that words appearing in similar contexts (distributional semantics) should have similar representations. The BGE model specifically uses a **contrastive learning** objective: given a query and a positive passage, maximize their similarity while minimizing similarity with negative (unrelated) passages.

The practical consequence for RAG is enormous: a user asking *"What triggers ocean acidification?"* will successfully retrieve a chunk about *"CO₂ absorption lowering seawater pH"* — even though no words overlap between the query and the answer.

In [ ]:
# ============================================================
# Cell: Demonstrate semantic vs lexical gap
# ============================================================
# These pairs have ZERO word overlap but high semantic similarity
semantic_pairs = [
    ("The automobile ran out of fuel", "The car had no gas left"),
    ("She was elated about the promotion", "She felt joyful about her career advancement"),
    ("The medication reduced inflammation", "The drug decreased swelling"),
]

# These pairs share words but differ in meaning
lexical_traps = [
    ("The bank of the river was muddy", "The bank increased interest rates"),
    ("He is a Python developer", "The python slithered through grass"),
    ("Apple released a new product", "I ate a delicious apple"),
]

print("=== High semantic similarity (zero word overlap) ===")
for a, b in semantic_pairs:
    emb_a, emb_b = embed_model.encode([a, b])
    sim = cosine_similarity(emb_a, emb_b)
    print(f"  sim = {sim:.4f}  |  '{a}' ↔ '{b}'")

print("\n=== Same words, different meanings (polysemy) ===")
for a, b in lexical_traps:
    emb_a, emb_b = embed_model.encode([a, b])
    sim = cosine_similarity(emb_a, emb_b)
    print(f"  sim = {sim:.4f}  |  '{a}' ↔ '{b}'")

print("\nEmbeddings correctly distinguish meaning even when words are identical!")

## 3.4 — Inside a Sentence Transformer: BGE Architecture

The BGE (BAAI General Embedding) model we're using is built on top of BERT. Understanding its architecture helps us reason about its strengths and limitations:

1. **Tokenization:** Input text is split into WordPiece tokens. The special token `[CLS]` is prepended. For BGE, the maximum sequence length is 512 tokens.

2. **Transformer Encoding:** The tokens pass through 12 transformer layers (for the "base" variant). Each layer applies self-attention and feed-forward networks, progressively building contextual representations.

3. **Pooling:** The 512 × 768 matrix of token representations is collapsed to a single 768-dim vector. BGE uses `[CLS]` token pooling (the representation of the first token), though many other models use mean pooling.

4. **Normalization:** The output vector is L2-normalized to unit length, so cosine similarity reduces to simple dot product.

The training uses a **contrastive loss** on (query, positive passage, negative passage) triples. The model learns to minimize:

$$\mathcal{L} = -\log \frac{e^{\text{sim}(q, p^+) / \tau}}{e^{\text{sim}(q, p^+) / \tau} + \sum_{i} e^{\text{sim}(q, p_i^-) / \tau}}$$

where $\tau$ is a temperature parameter. This is the InfoNCE loss, the same objective used in CLIP and many other representation learning methods.

---
# Part 4 — Vector Similarity Search with FAISS

Now that we understand embeddings, we need an efficient way to find the most similar vectors. Enter **FAISS** — Facebook AI Similarity Search.

## 4.1 — Why We Need Specialized Search

Given a query vector $q$ and a database of $n$ document vectors, finding the $k$ nearest neighbors via brute force requires computing $n$ dot products — that's $O(n \cdot d)$ per query.

For a small dataset (thousands of documents), this is fine. But consider production scale:
- **Wikipedia:** ~21 million paragraphs → 21M × 768 = ~60 GB of vectors
- **Enterprise docs:** 100M+ chunks are common
- **Real-time requirement:** Retrieval must complete in <100ms

Brute-force search over 100M vectors would take seconds, not milliseconds. We need **Approximate Nearest Neighbor (ANN)** algorithms that trade a tiny amount of accuracy for massive speedups.

**FAISS** is the industry-standard library for this. Developed by Facebook AI Research, it provides:
- Multiple index types (flat, IVF, HNSW, PQ) for different speed/accuracy tradeoffs
- GPU-accelerated search
- Billion-scale support via composite indices
- Seamless integration with numpy arrays

In [ ]:
# ============================================================
# Cell: Build a FAISS index from scratch
# ============================================================
import faiss
import numpy as np

# Sample documents across diverse topics
documents = [
    "Global warming causes sea levels to rise due to thermal expansion and melting ice sheets",
    "Machine learning models require large amounts of training data to generalize well",
    "Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen",
    "Neural networks consist of layers of interconnected nodes that process information",
    "Carbon dioxide traps heat in the atmosphere through the greenhouse effect",
    "Deep learning is a subset of machine learning using multiple neural network layers",
    "Ocean acidification occurs when CO2 is absorbed by seawater, lowering its pH",
    "The transformer architecture uses self-attention mechanisms for sequence processing",
    "Deforestation reduces the planet's capacity to absorb carbon dioxide",
    "Gradient descent optimizes model parameters by minimizing a loss function",
    "Coral bleaching is caused by rising ocean temperatures stressing coral organisms",
    "Convolutional neural networks are specialized for processing grid-like data such as images",
    "Methane is a greenhouse gas with 80 times the warming potential of CO2 over 20 years",
    "Backpropagation computes gradients by applying the chain rule through network layers",
    "Renewable energy sources include solar, wind, hydroelectric, and geothermal power",
]

print(f"Number of documents: {len(documents)}")

# Encode all documents
doc_embeddings = embed_model.encode(documents)
doc_embeddings = np.array(doc_embeddings).astype('float32')
print(f"Embeddings shape: {doc_embeddings.shape}")  # (15, 768)

# Normalize for cosine similarity
faiss.normalize_L2(doc_embeddings)

# Build the index (Inner Product ≡ cosine similarity after normalization)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"\nFAISS index built successfully:")
print(f"  Index type: IndexFlatIP (exact inner product search)")
print(f"  Vectors stored: {index.ntotal}")
print(f"  Dimension: {dimension}")
print(f"  Memory: ~{index.ntotal * dimension * 4 / 1024:.1f} KB")

In [ ]:
# ============================================================
# Cell: Query the FAISS index
# ============================================================
def search_index(query, index, documents, k=3):
    """Search the FAISS index and return top-k results."""
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    results = []
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'index': int(idx),
            'text': documents[idx]
        })
    return results

# Test with different queries
test_queries = [
    "What causes climate change?",
    "How do neural networks learn?",
    "What is photosynthesis?",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = search_index(query, index, documents, k=3)
    for r in results:
        print(f"  [{r['rank']}] score={r['score']:.4f} | doc[{r['index']}]: {r['text'][:80]}...")

## 4.2 — FAISS Index Types: A Practical Guide

FAISS provides several index types optimized for different scenarios:

| Index | Search Type | Time Complexity | Memory | Best For |
|---|---|---|---|---|
| **IndexFlatIP** | Exact | $O(n \cdot d)$ | $O(n \cdot d)$ | <100K vectors, accuracy-critical |
| **IndexFlatL2** | Exact (L2) | $O(n \cdot d)$ | $O(n \cdot d)$ | Same, when using L2 distance |
| **IndexIVFFlat** | Approximate | $O(\sqrt{n} \cdot d)$ | $O(n \cdot d)$ | 100K–10M vectors |
| **IndexIVFPQ** | Approximate + compressed | $O(\sqrt{n} \cdot m)$ | $O(n \cdot m)$ | 10M–1B vectors |
| **IndexHNSWFlat** | Approximate (graph) | $O(\log n \cdot d)$ | $O(n \cdot (d + M))$ | Low-latency requirement |

For this notebook, we use **IndexFlatIP** (exact inner-product search) because our dataset is small. In production with millions of chunks, you'd switch to IVF or HNSW.

**Key insight:** After L2-normalizing all vectors, inner product (IP) is equivalent to cosine similarity:

$$\text{cos}(a, b) = \frac{a \cdot b}{\|a\| \|b\|} \xrightarrow{\|a\|=\|b\|=1} a \cdot b$$

That's why we normalize first and then use `IndexFlatIP` — it gives us cosine similarity search with the speed of simple dot products.

In [ ]:
# ============================================================
# Cell: Compare brute-force vs FAISS timing
# ============================================================
import time

# Generate a larger dataset for timing comparison
np.random.seed(42)
n_vectors = 50_000
d = 768
large_dataset = np.random.randn(n_vectors, d).astype('float32')
faiss.normalize_L2(large_dataset)

query_vec = np.random.randn(1, d).astype('float32')
faiss.normalize_L2(query_vec)

# Method 1: NumPy brute force
start = time.time()
for _ in range(10):
    scores = np.dot(large_dataset, query_vec.T).flatten()
    top_k_np = np.argsort(scores)[-5:][::-1]
numpy_time = (time.time() - start) / 10

# Method 2: FAISS flat index
flat_index = faiss.IndexFlatIP(d)
flat_index.add(large_dataset)

start = time.time()
for _ in range(10):
    distances, indices = flat_index.search(query_vec, 5)
faiss_time = (time.time() - start) / 10

print(f"Search over {n_vectors:,} vectors (dim={d}):")
print(f"  NumPy brute force: {numpy_time*1000:.2f} ms")
print(f"  FAISS IndexFlatIP: {faiss_time*1000:.2f} ms")
print(f"  Speedup: {numpy_time/faiss_time:.1f}x")
print(f"\n  Results match: {set(top_k_np) == set(indices[0])}")

# Clean up
del large_dataset, flat_index

---
# Part 5 — Text Chunking Strategies

Before documents can be embedded, they must be split into chunks. Chunking is deceptively important — poor chunking can make or break a RAG system. This section builds chunkers from scratch and analyzes their tradeoffs.

## 5.1 — Why Chunking Matters

Embedding models have a maximum input length (512 tokens for BGE). Even if we could embed entire documents, we shouldn't — for two reasons:

1. **Relevance Precision:** A 10-page document about climate change covers many subtopics. If we embed the whole thing as one vector, a query about "ocean acidification" would get a mediocre match — the document is *somewhat* about that topic but mostly about other things. Smaller chunks give sharper, more precise matches.

2. **Context Window Efficiency:** The retrieved text is injected into the LLM's prompt. If we retrieve 3 chunks of 5,000 words each, we've used 15,000 tokens of context for information that could be captured in 3 chunks of 500 words. Smaller chunks = less wasted context.

**The chunking tradeoff:**
- **Too small** (50 words): Loses context. "It increased by 40%" is meaningless without knowing what "it" refers to.
- **Too large** (2000 words): Dilutes relevance. Contains many subtopics, most unrelated to the query.
- **Sweet spot** (200–500 words): Preserves enough context while maintaining topical focus.

The optimal chunk size depends on your domain, document structure, and query patterns. There's no universal answer — but there are principled strategies.

In [ ]:
# ============================================================
# Cell: Fixed-size character chunking
# ============================================================
def chunk_text_fixed(text, chunk_size=500, overlap=50):
    """
    Split text into fixed-size chunks with overlap.

    Args:
        text: Input text to chunk
        chunk_size: Maximum characters per chunk
        overlap: Number of characters to overlap between chunks

    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if c]  # Remove empty chunks

# Demonstrate
sample_text = """Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, such as through variations in the solar cycle, but since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil and gas.

Burning fossil fuels generates greenhouse gas emissions that act like a blanket wrapped around the Earth, trapping the sun's heat and raising temperatures. Examples of greenhouse gas emissions that are causing climate change include carbon dioxide and methane. These come from using gasoline for driving a car or coal for heating a building, for example. Clearing land and forests can also release carbon dioxide. Agriculture and landfills are key sources of methane emissions.

The effects of climate change are already being felt around the world. Temperatures are rising, sea levels are climbing, and extreme weather events are becoming more frequent. These changes threaten food production, water supplies, and human health."""

chunks = chunk_text_fixed(sample_text, chunk_size=300, overlap=50)
print(f"Original text length: {len(sample_text)} characters")
print(f"Number of chunks: {len(chunks)}")
print(f"Chunk size / overlap: 300 / 50\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)
    print()

In [ ]:
# ============================================================
# Cell: Recursive text chunking (split by structure)
# ============================================================
def chunk_text_recursive(text, chunk_size=500, overlap=50,
                         separators=None):
    """
    Recursively split text using a hierarchy of separators.

    Tries to split on the most meaningful boundary first
    (paragraphs), then falls back to sentences, then words,
    then characters.

    This preserves document structure better than fixed-size
    chunking.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    # Try each separator in order of preference
    for sep in separators:
        if sep == "":
            # Last resort: character-level split
            return chunk_text_fixed(text, chunk_size, overlap)

        parts = text.split(sep)
        if len(parts) <= 1:
            continue  # This separator doesn't split the text

        # Merge parts into chunks that fit within chunk_size
        chunks = []
        current = ""
        for part in parts:
            candidate = current + sep + part if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current.strip())
                # If a single part exceeds chunk_size, recurse
                if len(part) > chunk_size:
                    sub_chunks = chunk_text_recursive(
                        part, chunk_size, overlap,
                        separators[separators.index(sep)+1:]
                    )
                    chunks.extend(sub_chunks)
                    current = ""
                else:
                    current = part
        if current:
            chunks.append(current.strip())

        # Add overlap between chunks
        if overlap > 0 and len(chunks) > 1:
            overlapped = [chunks[0]]
            for i in range(1, len(chunks)):
                prev_tail = chunks[i-1][-overlap:]
                overlapped.append(prev_tail + " " + chunks[i])
            chunks = overlapped

        return [c for c in chunks if c]

    return [text.strip()] if text.strip() else []

# Compare both methods
print("=== Fixed-size chunking (300 chars) ===")
fixed_chunks = chunk_text_fixed(sample_text, 300, 50)
for i, c in enumerate(fixed_chunks):
    print(f"  Chunk {i}: {len(c)} chars — '{c[:60]}...'")

print(f"\n=== Recursive chunking (300 chars) ===")
recursive_chunks = chunk_text_recursive(sample_text, 300, 50)
for i, c in enumerate(recursive_chunks):
    print(f"  Chunk {i}: {len(c)} chars — '{c[:60]}...'")

print(f"\nFixed chunks: {len(fixed_chunks)}, Recursive chunks: {len(recursive_chunks)}")
print("Note: Recursive chunking respects paragraph boundaries!")

## 5.2 — The Chunk Size Tradeoff: An Empirical Analysis

Chunk size directly affects both retrieval quality and downstream generation quality. Let's visualize this tradeoff by embedding the same content at different granularities and measuring how well we can retrieve the right information.

**Small chunks (100 chars):** High precision but risk losing context. The chunk *"lowering its pH"* matches "ocean acidification" perfectly, but without surrounding context, it's not useful to the LLM.

**Large chunks (1000 chars):** Include more context but reduce precision. A chunk covering both ocean acidification AND coral bleaching will match queries about either topic — but with a lower score than a focused chunk.

The sweet spot depends on:
- **Query type:** Factoid questions → smaller chunks; complex reasoning → larger chunks
- **Document structure:** Well-structured documents (headers, paragraphs) → recursive chunking
- **Embedding model:** Models with longer max length (e.g., 8192 tokens) can handle larger chunks

In [ ]:
# ============================================================
# Cell: Chunk size quality analysis
# ============================================================
long_text = """Ocean acidification is the ongoing decrease in the pH of the Earth's ocean, caused by the uptake of carbon dioxide from the atmosphere. About 30% of the CO2 released by human activities is absorbed by the ocean, where it reacts with water to form carbonic acid. This process has caused ocean surface pH to drop from approximately 8.2 to 8.1 since the beginning of the Industrial Revolution. While this might seem like a small change, the pH scale is logarithmic, so this represents a roughly 26% increase in acidity.

The consequences for marine life are severe. Many organisms, particularly those that build calcium carbonate shells or skeletons, are directly threatened. Coral reefs, which support roughly 25% of all marine species, are especially vulnerable. As water becomes more acidic, it becomes harder for corals, mollusks, and certain plankton species to build and maintain their calcium carbonate structures.

Ocean acidification also disrupts the behavior of marine organisms. Studies have shown that fish in more acidic waters exhibit altered behavior, including reduced ability to detect predators, impaired decision-making, and changes in social behavior. The impacts cascade through food webs, potentially affecting fisheries that billions of people depend on for nutrition and livelihood."""

query = "What is ocean acidification and what are its effects?"

chunk_sizes = [100, 200, 400, 800]
print(f"Query: '{query}'\n")

for cs in chunk_sizes:
    chunks = chunk_text_fixed(long_text, chunk_size=cs, overlap=30)
    chunk_embeddings = embed_model.encode(chunks)
    chunk_embeddings = np.array(chunk_embeddings).astype('float32')
    query_emb = embed_model.encode([query]).astype('float32')

    # Compute cosine similarities
    norms_c = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
    norms_q = np.linalg.norm(query_emb, axis=1, keepdims=True)
    similarities = (chunk_embeddings @ query_emb.T) / (norms_c * norms_q.T)

    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx][0]

    print(f"Chunk size={cs:>4}  |  {len(chunks)} chunks  |  best_score={best_score:.4f}")
    print(f"  Best chunk: '{chunks[best_idx][:100]}...'\n")

In [ ]:
# ============================================================
# Cell: Sentence-aware chunking
# ============================================================
import re

def chunk_text_sentences(text, max_chunk_size=500, overlap_sentences=1):
    """
    Split text into chunks at sentence boundaries.

    This ensures no sentence is cut in half, preserving
    readability and semantic coherence.
    """
    # Split into sentences (handles ., !, ?)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current_sentences = []
    current_length = 0

    for sentence in sentences:
        if current_length + len(sentence) > max_chunk_size and current_sentences:
            chunks.append(" ".join(current_sentences))
            # Keep overlap_sentences from the end for context
            current_sentences = current_sentences[-overlap_sentences:]
            current_length = sum(len(s) for s in current_sentences)

        current_sentences.append(sentence)
        current_length += len(sentence)

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

# Compare all three methods
print("=== Sentence-aware chunking (500 chars, 1 sentence overlap) ===")
sent_chunks = chunk_text_sentences(long_text, max_chunk_size=400, overlap_sentences=1)
for i, c in enumerate(sent_chunks):
    print(f"  Chunk {i} ({len(c)} chars): '{c[:80]}...'")
    print()

print(f"Total chunks: {len(sent_chunks)}")
print("Note: Every chunk ends at a sentence boundary — no mid-sentence cuts!")

---
# Part 6 — Building the Complete RAG Pipeline

Now we bring everything together. We'll create a synthetic knowledge base, chunk it, embed it, index it, and build a complete question-answering system — all from scratch.

## 6.1 — The Synthetic Knowledge Base

Instead of loading external files, we define a comprehensive knowledge base inline. This makes the notebook fully self-contained and reproducible. Our topic: **Understanding Climate Change** — the same topic as the original notebook, but with richer content we fully control.

In [ ]:
# ============================================================
# Cell: Define the synthetic knowledge base
# ============================================================

KNOWLEDGE_BASE = [
    # --- Section 1: Fundamentals of Climate Change ---
    """Climate change refers to significant, long-term changes in the global climate. The global climate is the connected system of sun, earth, and oceans, wind, rain, and snow, forests, deserts, and savannas, and everything people do. The climate of a place, say New York, can be described as its rainfall, changing temperatures during the year, and so on. But the global climate is more than the average of the climates of specific places. The Earth's average surface temperature has increased by approximately 1.1 degrees Celsius since the pre-industrial era (1850-1900), with most of the warming occurring in the past 50 years. This rate of warming is unprecedented in at least the last 2,000 years of Earth's history.""",

    """The greenhouse effect is the fundamental mechanism driving climate change. When sunlight reaches the Earth's surface, some of it is reflected back toward space as infrared radiation (heat). Greenhouse gases in the atmosphere trap this heat, preventing it from escaping to space. This process is natural and essential for life — without the greenhouse effect, Earth's average temperature would be about -18°C instead of the current 15°C. However, human activities have dramatically increased the concentration of greenhouse gases, enhancing this natural effect and causing additional warming. The primary greenhouse gases are carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and fluorinated gases.""",

    """Carbon dioxide is the most significant greenhouse gas produced by human activities, accounting for about 75% of global greenhouse gas emissions. The atmospheric concentration of CO2 has risen from approximately 280 parts per million (ppm) in pre-industrial times to over 420 ppm today — a 50% increase. The primary sources of CO2 emissions are burning fossil fuels (coal, oil, and natural gas) for energy, which accounts for about 73% of emissions, deforestation and land-use changes (11%), and industrial processes like cement production (5%). The remaining 11% comes from agriculture and waste management.""",

    """Methane is the second most important greenhouse gas, responsible for about 16% of global warming. Although methane exists in the atmosphere in much smaller concentrations than CO2, it is approximately 80 times more potent as a warming agent over a 20-year period. Major sources of methane include agriculture (especially rice paddies and livestock digestion), fossil fuel extraction and processing (natural gas leaks, coal mining), landfills and waste treatment, and wetlands. Methane concentrations have more than doubled since pre-industrial times, from about 700 parts per billion to over 1,900 ppb.""",

    # --- Section 2: Impacts on Natural Systems ---
    """Rising global temperatures are causing profound changes to Earth's cryosphere — the frozen parts of the planet. Arctic sea ice has declined by approximately 13% per decade since 1979, with summer ice extent reaching record lows repeatedly. The Greenland and Antarctic ice sheets are losing mass at accelerating rates, contributing to sea level rise. Glaciers worldwide are retreating; over the past two decades, glaciers have lost approximately 267 billion tonnes of ice per year. Permafrost — permanently frozen ground in Arctic regions — is thawing, releasing stored carbon dioxide and methane, which creates a dangerous positive feedback loop that accelerates warming.""",

    """Sea level rise is one of the most consequential impacts of climate change. Global mean sea level has risen approximately 21-24 centimeters since 1880, with the rate of rise accelerating in recent decades. Current projections suggest sea levels could rise by 0.3 to 1.0 meters by 2100, depending on emission scenarios. Sea level rise is driven by two primary mechanisms: thermal expansion (water expands as it warms, contributing about 42% of observed rise) and the melting of land-based ice (contributing about 58%). Even modest sea level rise threatens coastal cities, low-lying island nations, and hundreds of millions of people living in coastal areas.""",

    """Ocean acidification is often called climate change's "evil twin." About 30% of the CO2 released by human activities is absorbed by the ocean, where it reacts with seawater to form carbonic acid. This has caused a 26% increase in ocean acidity since the Industrial Revolution. The consequences for marine ecosystems are severe: coral reefs, which support 25% of all marine species, are experiencing mass bleaching events and dissolution of their calcium carbonate structures. Shellfish, certain plankton species, and other calcifying organisms struggle to build and maintain their shells. These disruptions cascade through marine food webs, threatening fisheries that feed billions of people.""",

    """Climate change is fundamentally altering weather patterns worldwide. The frequency and intensity of extreme weather events — heat waves, heavy rainfall, droughts, and tropical cyclones — are increasing. Heat waves have become more frequent and longer-lasting; the number of heat wave days per year has tripled since the 1960s in many regions. Precipitation patterns are shifting: wet regions generally become wetter and dry regions drier, though with significant local variations. Tropical cyclones are becoming more intense (though not necessarily more frequent), with higher peak wind speeds and heavier rainfall due to warmer ocean surfaces and a more moisture-laden atmosphere.""",

    # --- Section 3: Impacts on Human Systems ---
    """Climate change poses severe threats to global food security. Rising temperatures reduce crop yields in tropical regions, where many of the world's poorest people live. For every degree of warming, global wheat yields are expected to decline by 6%, rice yields by 3.2%, and maize yields by 7.4%. Changes in precipitation patterns cause both floods and droughts, destroying harvests. Shifting growing seasons disrupt traditional farming practices. Meanwhile, CO2 fertilization — where higher CO2 levels boost plant growth — partially offsets yield losses for some crops, but this effect is diminishing and comes with reduced nutritional content (lower protein and micronutrient concentrations in grains).""",

    """The health impacts of climate change are wide-ranging and growing. Heat-related mortality is rising: extreme heat causes an estimated 489,000 deaths annually. Vector-borne diseases like malaria and dengue are expanding into previously temperate regions as warmer temperatures allow disease-carrying mosquitoes to thrive in new areas. Air quality is worsening due to increased ground-level ozone formation and wildfire smoke. Mental health impacts are increasingly recognized, including eco-anxiety, climate grief, and post-traumatic stress following extreme weather events. The WHO estimates that climate change will cause approximately 250,000 additional deaths per year between 2030 and 2050 from malnutrition, malaria, diarrhea, and heat stress alone.""",

    """Economic impacts of climate change are already substantial and projected to grow dramatically. The annual cost of climate-related disasters has increased from approximately $50 billion in the 1980s to over $200 billion in recent years. By 2100, unmitigated climate change could reduce global GDP by 10-23% compared to a scenario without warming. The economic burdens fall disproportionately on developing nations, which contribute least to emissions but are most vulnerable to climate impacts. Key economic risks include damage to infrastructure from extreme weather, reduced agricultural productivity, increased healthcare costs, disruption to supply chains, and loss of tourism revenue in affected regions.""",

    # --- Section 4: Solutions and Mitigation ---
    """Transitioning to renewable energy is the cornerstone of climate change mitigation. Solar and wind power have experienced dramatic cost reductions: solar photovoltaic costs have fallen by 89% since 2010, and onshore wind by 70%. Renewables now account for about 30% of global electricity generation. However, the transition must accelerate significantly to meet climate targets. Key challenges include intermittency (the sun doesn't always shine, wind doesn't always blow), the need for massive energy storage solutions (batteries, pumped hydro, hydrogen), upgrading electrical grid infrastructure, and phasing out fossil fuel subsidies which still total over $7 trillion annually globally when including implicit subsidies.""",

    """Carbon capture and storage (CCS) technologies aim to capture CO2 from emission sources or directly from the atmosphere and store it permanently underground. Current CCS capacity globally is about 45 million tonnes of CO2 per year, but the IPCC estimates we may need to capture 5-16 billion tonnes annually by 2050 to meet the 1.5°C target. Direct air capture (DAC) is a promising but expensive technology, currently costing $250-600 per tonne of CO2 removed. Nature-based solutions — reforestation, wetland restoration, soil carbon sequestration — offer lower-cost carbon removal but face challenges of permanence and scalability. A portfolio approach combining multiple methods is considered most viable.""",

    """International climate agreements provide the framework for global action. The Paris Agreement (2015) set the goal of limiting warming to well below 2°C, preferably 1.5°C, above pre-industrial levels. Under the agreement, countries submit Nationally Determined Contributions (NDCs) outlining their emission reduction plans. As of 2024, current NDCs are insufficient — they put the world on track for approximately 2.5-2.9°C of warming by 2100. The annual Conference of the Parties (COP) meetings serve as the primary venue for negotiating stronger commitments. Key ongoing debates include climate finance for developing nations, loss and damage compensation, and the pace of fossil fuel phase-out.""",

    """Individual and community actions play a significant role in addressing climate change, even though systemic changes are most critical. The average carbon footprint varies enormously by country: approximately 16 tonnes CO2 per person annually in the United States, compared to 1.9 tonnes in India. The highest-impact individual actions include adopting a plant-rich diet (reducing food-related emissions by up to 50%), reducing air travel, using public transportation or electric vehicles, improving home energy efficiency, and reducing consumption overall. Community-level actions such as local renewable energy cooperatives, urban tree planting programs, and advocacy for policy change amplify individual impact.""",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
for i, doc in enumerate(KNOWLEDGE_BASE):
    print(f"  Doc {i:>2}: {len(doc)} chars — '{doc[:70]}...'")

In [ ]:
# ============================================================
# Cell: Step 1 — Chunk all documents
# ============================================================
all_chunks = []
chunk_metadata = []  # Track which document each chunk came from

for doc_idx, doc in enumerate(KNOWLEDGE_BASE):
    doc_chunks = chunk_text_sentences(doc, max_chunk_size=400, overlap_sentences=1)
    for chunk in doc_chunks:
        all_chunks.append(chunk)
        chunk_metadata.append({
            'doc_index': doc_idx,
            'chunk_index': len(all_chunks) - 1
        })

print(f"Chunking complete:")
print(f"  Documents: {len(KNOWLEDGE_BASE)}")
print(f"  Total chunks: {len(all_chunks)}")
print(f"  Avg chunk length: {np.mean([len(c) for c in all_chunks]):.0f} chars")
print(f"  Min chunk length: {min(len(c) for c in all_chunks)} chars")
print(f"  Max chunk length: {max(len(c) for c in all_chunks)} chars")
print(f"\nSample chunks:")
for i in [0, len(all_chunks)//2, len(all_chunks)-1]:
    print(f"  Chunk {i} (doc {chunk_metadata[i]['doc_index']}): '{all_chunks[i][:80]}...'")

In [ ]:
# ============================================================
# Cell: Step 2 — Embed all chunks
# ============================================================
import time

start_time = time.time()
chunk_embeddings = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=32
)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')
embed_time = time.time() - start_time

print(f"\nEmbedding complete:")
print(f"  Shape: {chunk_embeddings.shape}")
print(f"  Time: {embed_time:.2f}s ({len(all_chunks)/embed_time:.0f} chunks/sec)")
print(f"  Memory: {chunk_embeddings.nbytes / 1024:.1f} KB")

In [ ]:
# ============================================================
# Cell: Step 3 — Build FAISS index
# ============================================================
# Normalize embeddings for cosine similarity
faiss.normalize_L2(chunk_embeddings)

# Create and populate the index
dimension = chunk_embeddings.shape[1]
rag_index = faiss.IndexFlatIP(dimension)
rag_index.add(chunk_embeddings)

print(f"FAISS index ready:")
print(f"  Vectors: {rag_index.ntotal}")
print(f"  Dimension: {dimension}")
print(f"  Index type: Flat Inner Product (exact cosine similarity)")

In [ ]:
# ============================================================
# Cell: Step 4 — Define retrieval function
# ============================================================
def retrieve(query, k=3):
    """
    Retrieve the top-k most relevant chunks for a query.

    Args:
        query: Natural language question
        k: Number of chunks to retrieve

    Returns:
        List of dicts with 'chunk', 'score', 'index', 'doc_index'
    """
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding)

    distances, indices = rag_index.search(query_embedding, k)

    results = []
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            'rank': rank + 1,
            'chunk': all_chunks[idx],
            'score': float(score),
            'index': int(idx),
            'doc_index': chunk_metadata[idx]['doc_index']
        })
    return results

# Test retrieval
test_q = "What causes ocean acidification?"
results = retrieve(test_q, k=3)
print(f"Query: '{test_q}'\n")
for r in results:
    print(f"  [{r['rank']}] score={r['score']:.4f} (doc {r['doc_index']}, chunk {r['index']})")
    print(f"      '{r['chunk'][:120]}...'\n")

In [ ]:
# ============================================================
# Cell: Step 5+6 — Build the complete RAG pipeline
# ============================================================
def rag_query(question, k=3, verbose=True):
    """
    Complete RAG pipeline: Retrieve → Augment → Generate.

    Args:
        question: User's natural language question
        k: Number of chunks to retrieve
        verbose: Whether to print retrieval details

    Returns:
        Tuple of (answer_text, retrieved_chunks)
    """
    # Step 1: Retrieve relevant chunks
    retrieved = retrieve(question, k=k)

    if verbose:
        print(f"📥 Retrieved {len(retrieved)} chunks:")
        for r in retrieved:
            print(f"   [{r['rank']}] score={r['score']:.4f} — '{r['chunk'][:80]}...'")
        print()

    # Step 2: Build context from retrieved chunks
    context = "\n\n".join([
        f"[Source {i+1}] {r['chunk']}"
        for i, r in enumerate(retrieved)
    ])

    # Step 3: Construct the augmented prompt
    prompt = f"""Based on the following context, answer the question accurately and concisely. Use only information from the provided context. If the context doesn't contain enough information to fully answer the question, say so explicitly.

Context:
{context}

Question: {question}

Answer:"""

    # Step 4: Generate answer
    if verbose:
        print("🤖 Generating answer...")
    answer = generate(prompt, max_new_tokens=300)

    return answer, retrieved

# Test the complete pipeline
print("=" * 70)
print("RAG PIPELINE TEST")
print("=" * 70)
answer, chunks = rag_query("What is ocean acidification and what are its effects on marine life?")
print(f"\n📝 Answer:\n{answer}")

In [ ]:
# ============================================================
# Cell: Run multiple diverse queries
# ============================================================
test_questions = [
    "How much have solar energy costs decreased in recent years?",
    "What are the health impacts of climate change?",
    "What is the Paris Agreement and is it working?",
    "How does methane compare to CO2 as a greenhouse gas?",
    "What is happening to Arctic sea ice?",
]

for i, question in enumerate(test_questions):
    print(f"\n{'='*70}")
    print(f"Question {i+1}: {question}")
    print('='*70)
    answer, retrieved = rag_query(question, k=3, verbose=False)
    print(f"\nTop retrieval scores: {[f'{r["score"]:.3f}' for r in retrieved]}")
    print(f"\nAnswer: {answer[:500]}")

In [ ]:
# ============================================================
# Cell: Hallucination demo — with vs without retrieval
# ============================================================
def generate_without_rag(question):
    """Ask the LLM directly, with no retrieved context."""
    prompt = f"""Answer the following question. If you're unsure, say so.

Question: {question}

Answer:"""
    return generate(prompt, max_new_tokens=300)

# Use a question where our knowledge base has specific data
test_q = "What percentage of global greenhouse gas emissions does carbon dioxide account for?"

print("=" * 70)
print("RAG vs NO-RAG COMPARISON")
print("=" * 70)
print(f"\nQuestion: {test_q}")

print("\n--- WITH RAG (grounded in knowledge base) ---")
rag_answer, rag_chunks = rag_query(test_q, k=3, verbose=False)
print(f"Sources used: {[r['doc_index'] for r in rag_chunks]}")
print(f"Answer: {rag_answer}")

print("\n--- WITHOUT RAG (LLM memory only) ---")
no_rag_answer = generate_without_rag(test_q)
print(f"Answer: {no_rag_answer}")

print("\n--- ANALYSIS ---")
print("The RAG answer should cite specific numbers from our knowledge base (75%).")
print("The no-RAG answer relies on the model's training data and may differ or be less precise.")

---
# Part 7 — Evaluating RAG Systems

How do we know our RAG system is working well? Evaluation is critical but often overlooked. RAG evaluation has two independent dimensions: **retrieval quality** (did we find the right chunks?) and **generation quality** (did the LLM produce a correct answer?).

## 7.1 — Retrieval Metrics

Retrieval metrics measure how well the retriever finds relevant documents. We need a ground truth: for each query, we define which chunks *should* be retrieved.

**Precision@K:** Of the K chunks we retrieved, how many were relevant?
$$\text{Precision@K} = \frac{|\text{retrieved}_K \cap \text{relevant}|}{K}$$

**Recall@K:** Of all relevant chunks, how many did we retrieve?
$$\text{Recall@K} = \frac{|\text{retrieved}_K \cap \text{relevant}|}{|\text{relevant}|}$$

**Mean Reciprocal Rank (MRR):** How high does the first relevant result appear?
$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

where $\text{rank}_i$ is the position of the first relevant result for query $i$.

**NDCG@K (Normalized Discounted Cumulative Gain):** Considers not just relevance but ranking position — higher-ranked relevant results are weighted more heavily. This is the gold standard for ranking evaluation.

In [ ]:
# ============================================================
# Cell: Implement retrieval evaluation metrics
# ============================================================
def precision_at_k(retrieved_indices, relevant_indices, k):
    """Fraction of retrieved items that are relevant."""
    retrieved_k = set(list(retrieved_indices)[:k])
    relevant = set(relevant_indices)
    if k == 0:
        return 0.0
    return len(retrieved_k & relevant) / k

def recall_at_k(retrieved_indices, relevant_indices, k):
    """Fraction of relevant items that were retrieved."""
    retrieved_k = set(list(retrieved_indices)[:k])
    relevant = set(relevant_indices)
    if len(relevant) == 0:
        return 0.0
    return len(retrieved_k & relevant) / len(relevant)

def reciprocal_rank(retrieved_indices, relevant_indices):
    """Reciprocal of the rank of the first relevant result."""
    relevant = set(relevant_indices)
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved_indices, relevant_indices, k):
    """Normalized Discounted Cumulative Gain at K."""
    import math
    relevant = set(relevant_indices)

    # DCG: sum of relevance / log2(rank+1)
    dcg = 0.0
    for rank, idx in enumerate(list(retrieved_indices)[:k], start=1):
        rel = 1.0 if idx in relevant else 0.0
        dcg += rel / math.log2(rank + 1)

    # Ideal DCG: all relevant items at top
    ideal_dcg = 0.0
    n_relevant = min(len(relevant), k)
    for rank in range(1, n_relevant + 1):
        ideal_dcg += 1.0 / math.log2(rank + 1)

    if ideal_dcg == 0:
        return 0.0
    return dcg / ideal_dcg

# Demonstrate with a simple example
retrieved = [3, 7, 1, 5, 9]
relevant  = [1, 3, 8]

print("Retrieval evaluation example:")
print(f"  Retrieved indices: {retrieved}")
print(f"  Relevant indices:  {list(relevant)}")
print(f"\n  Precision@1: {precision_at_k(retrieved, relevant, 1):.3f}")
print(f"  Precision@3: {precision_at_k(retrieved, relevant, 3):.3f}")
print(f"  Precision@5: {precision_at_k(retrieved, relevant, 5):.3f}")
print(f"  Recall@3:    {recall_at_k(retrieved, relevant, 3):.3f}")
print(f"  Recall@5:    {recall_at_k(retrieved, relevant, 5):.3f}")
print(f"  MRR:         {reciprocal_rank(retrieved, relevant):.3f}")
print(f"  NDCG@3:      {ndcg_at_k(retrieved, relevant, 3):.3f}")
print(f"  NDCG@5:      {ndcg_at_k(retrieved, relevant, 5):.3f}")

In [ ]:
# ============================================================
# Cell: Create evaluation test set with ground truth
# ============================================================
# Each test case: (query, list of relevant document indices)
# Document indices refer to our KNOWLEDGE_BASE (0-14)
eval_test_set = [
    {
        "query": "What is the greenhouse effect?",
        "relevant_docs": [1],  # Doc about greenhouse effect
        "description": "Direct factual question about the greenhouse mechanism"
    },
    {
        "query": "How much CO2 is in the atmosphere now?",
        "relevant_docs": [2],  # Doc about CO2 concentrations
        "description": "Specific numeric question about CO2 levels"
    },
    {
        "query": "What are the effects of ocean acidification on coral reefs?",
        "relevant_docs": [6],  # Doc about ocean acidification
        "description": "Question about specific environmental impact"
    },
    {
        "query": "How is methane produced and why is it dangerous?",
        "relevant_docs": [3],  # Doc about methane
        "description": "Question about methane sources and impact"
    },
    {
        "query": "What is happening to global sea levels?",
        "relevant_docs": [5],  # Doc about sea level rise
        "description": "Question about sea level changes"
    },
    {
        "query": "How does climate change affect food production?",
        "relevant_docs": [8],  # Doc about food security
        "description": "Question about agricultural impacts"
    },
    {
        "query": "What are the goals of the Paris Agreement?",
        "relevant_docs": [13],  # Doc about international agreements
        "description": "Question about climate policy"
    },
    {
        "query": "How has the cost of renewable energy changed?",
        "relevant_docs": [11],  # Doc about renewable energy
        "description": "Question about energy economics"
    },
]

print(f"Evaluation test set: {len(eval_test_set)} queries")
for i, tc in enumerate(eval_test_set):
    print(f"  [{i}] '{tc['query'][:50]}...' → docs {tc['relevant_docs']}")

In [ ]:
# ============================================================
# Cell: Run evaluation over the test set
# ============================================================
def find_relevant_chunks(relevant_doc_indices, chunk_metadata):
    """Map relevant document indices to relevant chunk indices."""
    return [
        i for i, meta in enumerate(chunk_metadata)
        if meta['doc_index'] in relevant_doc_indices
    ]

print(f"{'Query':<55} {'P@1':>5} {'P@3':>5} {'R@3':>5} {'MRR':>5} {'NDCG@3':>7}")
print("-" * 85)

all_metrics = {'p1': [], 'p3': [], 'r3': [], 'mrr': [], 'ndcg3': []}

for tc in eval_test_set:
    query = tc['query']
    relevant_chunks = find_relevant_chunks(tc['relevant_docs'], chunk_metadata)

    # Retrieve
    results = retrieve(query, k=5)
    retrieved_indices = [r['index'] for r in results]

    # Compute metrics
    p1 = precision_at_k(retrieved_indices, relevant_chunks, 1)
    p3 = precision_at_k(retrieved_indices, relevant_chunks, 3)
    r3 = recall_at_k(retrieved_indices, relevant_chunks, 3)
    mrr = reciprocal_rank(retrieved_indices, relevant_chunks)
    ndcg3 = ndcg_at_k(retrieved_indices, relevant_chunks, 3)

    all_metrics['p1'].append(p1)
    all_metrics['p3'].append(p3)
    all_metrics['r3'].append(r3)
    all_metrics['mrr'].append(mrr)
    all_metrics['ndcg3'].append(ndcg3)

    short_q = query[:52] + "..." if len(query) > 52 else query
    print(f"{short_q:<55} {p1:>5.2f} {p3:>5.2f} {r3:>5.2f} {mrr:>5.2f} {ndcg3:>7.3f}")

print("-" * 85)
print(f"{'AVERAGE':<55} {np.mean(all_metrics['p1']):>5.2f} "
      f"{np.mean(all_metrics['p3']):>5.2f} {np.mean(all_metrics['r3']):>5.2f} "
      f"{np.mean(all_metrics['mrr']):>5.2f} {np.mean(all_metrics['ndcg3']):>7.3f}")

print(f"\nInterpretation:")
print(f"  • MRR = {np.mean(all_metrics['mrr']):.2f} means the first relevant result is typically at rank {1/np.mean(all_metrics['mrr']):.1f}")
print(f"  • P@3 = {np.mean(all_metrics['p3']):.2f} means {np.mean(all_metrics['p3'])*100:.0f}% of top-3 results are relevant")
print(f"  • R@3 = {np.mean(all_metrics['r3']):.2f} means we find {np.mean(all_metrics['r3'])*100:.0f}% of all relevant chunks in top-3")

---
# Part 8 — RAG vs No-RAG: A Systematic Comparison

The best way to appreciate RAG's value is to see it in action alongside bare LLM responses. We'll compare answers on questions where our knowledge base has *specific* data points the LLM may not have memorized precisely.

In [ ]:
# ============================================================
# Cell: Side-by-side comparison on multiple questions
# ============================================================
comparison_questions = [
    "By how much has ocean acidity increased since the Industrial Revolution?",
    "How many deaths per year does extreme heat cause?",
    "What is the current global capacity of carbon capture and storage?",
    "What is the cost range for direct air capture of CO2?",
]

for q in comparison_questions:
    print("\n" + "=" * 70)
    print(f"Q: {q}")
    print("=" * 70)

    # RAG answer
    rag_answer, retrieved = rag_query(q, k=3, verbose=False)
    print(f"\n🟢 RAG Answer (grounded in retrieved context):")
    print(f"   Retrieval scores: {[f'{r["score"]:.3f}' for r in retrieved]}")
    print(f"   {rag_answer[:400]}")

    # No-RAG answer
    no_rag_answer = generate_without_rag(q)
    print(f"\n🔴 No-RAG Answer (LLM memory only):")
    print(f"   {no_rag_answer[:400]}")

In [ ]:
# ============================================================
# Cell: Latency comparison
# ============================================================
import time

test_question = "What are the main sources of methane emissions?"

# Measure RAG latency (retrieval + generation)
start = time.time()
_ = retrieve(test_question, k=3)
retrieval_time = time.time() - start

start = time.time()
rag_answer, _ = rag_query(test_question, k=3, verbose=False)
rag_total_time = time.time() - start

# Measure no-RAG latency (generation only)
start = time.time()
no_rag_answer = generate_without_rag(test_question)
no_rag_time = time.time() - start

print(f"Latency comparison for: '{test_question}'")
print(f"\n  Retrieval only:  {retrieval_time*1000:>8.1f} ms")
print(f"  RAG total:       {rag_total_time*1000:>8.1f} ms  (retrieval + generation)")
print(f"  No-RAG:          {no_rag_time*1000:>8.1f} ms  (generation only)")
print(f"  RAG overhead:    {(rag_total_time - no_rag_time)*1000:>8.1f} ms")
print(f"\nNote: Retrieval is typically <50ms — the LLM generation dominates latency.")
print(f"The additional context tokens from RAG may slightly increase generation time.")

## 8.1 — When RAG Is and Isn't Needed

RAG isn't always the right tool. Understanding when to use it — and when it's overkill — is an important engineering skill.

**RAG excels when:**
- The question requires facts that change over time (news, stock prices, recent events)
- The domain has specialized knowledge not well-represented in LLM training data
- Attribution and source citation are requirements
- Factual accuracy is critical (medical, legal, financial domains)
- You need to ground the model in your organization's specific documents

**RAG is overkill when:**
- The question is about general knowledge the LLM already knows well ("What is photosynthesis?")
- The task is creative (writing fiction, brainstorming ideas)
- The task requires reasoning, not fact retrieval (math, logic puzzles)
- Response latency is extremely critical and you can tolerate less accuracy

**RAG fails when:**
- The retriever can't find relevant documents (vocabulary mismatch, poor chunking)
- The relevant information is spread across many chunks that can't all fit in context
- The question requires multi-hop reasoning across multiple documents
- The knowledge base doesn't contain the answer at all

---
# Part 9 — Synthesis & Production Considerations

Let's consolidate everything we've learned and discuss how this educational pipeline maps to production systems.

## 9.1 — The Complete RAG Pipeline (Recap)

We built a complete RAG system from scratch using these components:

```
┌─────────────────────────────────────────────────────────────────────┐
│                   OFFLINE PIPELINE (Index Time)                     │
│                                                                     │
│  Raw Documents                                                      │
│       │                                                             │
│       ▼                                                             │
│  ┌─────────────┐    ┌──────────────┐    ┌──────────────────┐       │
│  │  CHUNKING   │───▶│  EMBEDDING   │───▶│  FAISS INDEXING  │       │
│  │             │    │              │    │                  │       │
│  │ Sentence-   │    │ BGE-base     │    │ IndexFlatIP     │       │
│  │ aware split │    │ 768 dims     │    │ L2-normalized   │       │
│  │ 400 chars   │    │ batch=32     │    │ cosine sim      │       │
│  └─────────────┘    └──────────────┘    └──────────────────┘       │
│                                                                     │
├─────────────────────────────────────────────────────────────────────┤
│                   ONLINE PIPELINE (Query Time)                      │
│                                                                     │
│  User Query ──▶ Embed Query ──▶ FAISS Search (top-k)               │
│       │                              │                              │
│       │                              ▼                              │
│       │                        Retrieved Chunks                     │
│       │                              │                              │
│       ▼                              ▼                              │
│  ┌────────────────────────────────────────────┐                     │
│  │  PROMPT CONSTRUCTION                       │                     │
│  │  "Based on the following context..."       │                     │
│  │  [Source 1] chunk_text_1                    │                     │
│  │  [Source 2] chunk_text_2                    │                     │
│  │  [Source 3] chunk_text_3                    │                     │
│  │  Question: user_query                      │                     │
│  └────────────────────────────────────────────┘                     │
│       │                                                             │
│       ▼                                                             │
│  ┌────────────────────────────────────────────┐                     │
│  │  LLM GENERATION (Qwen3-8B)               │                     │
│  │  Generates grounded answer                  │                     │
│  └────────────────────────────────────────────┘                     │
│       │                                                             │
│       ▼                                                             │
│  Grounded Answer + Source Citations                                 │
└─────────────────────────────────────────────────────────────────────┘
```

Every component — chunking, embedding, indexing, retrieval, augmentation, generation — was implemented in plain Python with no framework dependencies.

## 9.2 — Scaling to Production

Our notebook pipeline works perfectly for learning. Here's how each component changes at production scale:

**Chunking:**
- Add metadata extraction (titles, headings, dates, authors)
- Use document-type-specific parsers (PDF, HTML, DOCX, etc.)
- Implement overlap strategies tuned to your query patterns
- Consider semantic chunking (split by topic, not just length)

**Embedding:**
- Batch embedding with GPU acceleration
- Pre-compute and cache embeddings
- Consider quantized embeddings (int8, binary) for memory savings
- Monitor embedding drift — retrain or switch models periodically

**Indexing:**
- Move from IndexFlatIP to IVF or HNSW for datasets > 100K chunks
- Use FAISS with GPU (faiss-gpu) for massive datasets
- Consider managed solutions: Pinecone, Weaviate, Qdrant, Milvus
- Implement incremental updates (add/remove documents without full rebuild)

**Retrieval:**
- Hybrid search: combine dense (embedding) + sparse (BM25) retrieval
- Reranking: use a cross-encoder to re-score top candidates
- Query expansion: generate multiple query variants to improve recall
- Metadata filtering: pre-filter by date, source, category before similarity search

**Generation:**
- Streaming output for better user experience
- Citation extraction: map answer sentences back to source chunks
- Confidence scoring: detect when retrieved context is insufficient
- Guardrails: prevent the model from contradicting retrieved evidence

## 9.3 — Common RAG Failure Modes

Understanding how RAG systems fail is as important as knowing how they work:

| Failure Mode | Symptom | Root Cause | Mitigation |
|---|---|---|---|
| **Retrieval Miss** | Correct answer exists in DB but isn't retrieved | Vocabulary mismatch between query and chunk | Query expansion, hybrid search |
| **Context Poisoning** | Retrieved chunks contain incorrect info | Outdated or wrong data in knowledge base | Data quality pipeline, freshness checks |
| **Lost in the Middle** | LLM ignores relevant info in middle of context | Known LLM limitation with long contexts | Rerank to put best chunks first, limit context length |
| **Cross-chunk Reasoning** | Answer requires combining info from multiple docs | Each chunk is retrieved independently | Multi-hop retrieval, document summaries |
| **Confidence Mismatch** | LLM sounds confident about wrong answer | LLM style training (always sound confident) | Add uncertainty prompting, verify against sources |
| **Chunk Boundary Split** | Key information split across two chunks | Fixed-size chunking cuts mid-sentence | Sentence-aware chunking with overlap |

The most impactful improvement for most RAG systems is better chunking and retrieval quality — not a more powerful LLM. A perfect retriever with a mediocre generator outperforms a mediocre retriever with a perfect generator.

## 9.4 — What's Next: Advanced RAG Techniques

This notebook covered the foundational RAG pipeline. The other notebooks in this series build on this foundation with advanced techniques:

1. **Chunking strategies** — Semantic chunking, proposition chunking, contextual chunk headers
2. **Query transformation** — HyDE (hypothetical document embeddings), query decomposition, step-back prompting
3. **Retrieval improvements** — Fusion retrieval, reranking, adaptive retrieval
4. **Architecture patterns** — Self-RAG, CRAG (Corrective RAG), Agentic RAG, RAPTOR
5. **Evaluation** — Automated RAG evaluation, feedback loops, reliability scoring
6. **Multimodal RAG** — Processing images, tables, and mixed-media documents

Each technique addresses one or more of the failure modes discussed above. The key insight is that RAG is not a single algorithm — it's a *design pattern* with many interchangeable components, each of which can be independently improved.

> **The best RAG system is the one whose failure modes you understand and have mitigated for your specific use case.**

---
## Summary

In this notebook, we built a complete RAG system from first principles:

| Step | What We Built | Key Concept |
|---|---|---|
| **Part 1** | Theoretical foundation | What RAG is and why it exists |
| **Part 2** | Embedding analysis | Text → dense vectors, cosine similarity |
| **Part 3** | FAISS vector search | Efficient nearest-neighbor retrieval |
| **Part 4** | Text chunking | Fixed, recursive, and sentence-aware splitting |
| **Part 5** | Complete pipeline | Retrieve → Augment → Generate |
| **Part 6** | Evaluation metrics | Precision, Recall, MRR, NDCG |
| **Part 7** | RAG vs No-RAG | Grounded vs hallucinated answers |
| **Part 8** | Production guide | Scaling, failure modes, and next steps |

**Every component was implemented in native Python** — no frameworks, no APIs, no black boxes. You now understand exactly how each piece works, which means you can debug, optimize, and extend any RAG system you encounter.

*This notebook is part of the [Castalia](https://github.com/AI2026) series on advanced NLP techniques.*